# Kombinierte Labels: CatBoost

**Hypothese:** CatBoost kann additive, nichtlineare Wechselwirkungen zwischen Rohspektren, AEZ und Month besser nutzen als die bisherigen Baum-Ensembles und nähert sich dadurch der kombinierten RBF-SVM an.

Die Hyperparameter werden ausschließlich auf den fünf gemeinsamen Trainings-Folds ausgewählt. Der gemeinsame Validierungsanteil wird danach genau einmal bewertet.

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd
from IPython.display import display

from awp2.data import combined_label
from awp2.experiment import RunConfig, TuneConfig, run, tune
from awp2.models import make_combined_catboost
from awp2.preprocessing import PreprocessingConfig

COMBINED_SVM_CV_BACC = 0.847
COMBINED_SVM_VAL_BACC = 0.866

catboost_preprocessing = PreprocessingConfig(scale=False)

## 1. Hyperparameter suchen

Die acht Kandidaten variieren die Baumtiefe, Lernrate und L2-Regularisierung. Klassengewichte, Seed und Iterationsbudget bleiben im gemeinsamen Factory-Builder fest; CatBoost benötigt keine Skalierung.

In [ ]:
tuned = tune(
    make_combined_catboost(),
    TuneConfig(
        name="combined_catboost_search",
        approach="combined",
        description=(
            "CatBoost auf kombinierten Crop/Stage-Labels: Tiefe, Lernrate und "
            "L2-Regularisierung vergleichen."
        ),
        preprocessing=catboost_preprocessing,
        param_grid={
            "estimator__depth": [4, 6],
            "estimator__learning_rate": [0.03, 0.1],
            "estimator__l2_leaf_reg": [3.0, 10.0],
        },
    ),
)

display(tuned.candidates.sort_values("rank"))
{
    "best_cv_score": tuned.best_score,
    "best_params": tuned.best_params,
    "tuning_run": tuned.run_id,
}

## 2. Einmal validieren

Die beste Konfiguration aus der CV wird genau einmal auf dem gemeinsamen Validierungsanteil geprüft. Die Entscheidung über ihre Hyperparameter bleibt dabei ausschließlich in der Trainings-CV.

In [ ]:
result = run(
    tuned.best_model,
    RunConfig(
        name="combined_catboost",
        approach="combined",
        description=(
            "Beste CatBoost-Konfiguration aus combined_catboost_search, einmal auf der "
            "Validierung bewertet."
        ),
        preprocessing=catboost_preprocessing,
        tuning_run=tuned.run_id,
    ),
)

result.metrics.model_dump()

## 3. Analyse

Die Tabelle trennt CV und Validierung. Die Verwechslungen beziehen sich auf die kombinierten Klassen und machen sichtbar, ob die stärksten Fehler eher Kultur, Stadium oder deren Paarung betreffen.

In [ ]:
from awp2.evaluation import plot_confusion_matrices
from awp2.plots import plot_class_recall, plot_top_confusions, plot_tuning_candidates

comparison = pd.DataFrame(
    {
        "CV-BAcc kombiniert": {
            "CatBoost": tuned.best_score,
            "RBF-SVM": COMBINED_SVM_CV_BACC,
        },
        "Validierungs-BAcc kombiniert": {
            "CatBoost": result.metrics.bacc_combined,
            "RBF-SVM": COMBINED_SVM_VAL_BACC,
        },
    }
)
display(comparison)

top_confusions = (
    pd.DataFrame(
        {"Wahr": combined_label(result.y_val), "Vorhergesagt": combined_label(result.y_pred)}
    )
    .query("Wahr != Vorhergesagt")
    .value_counts()
    .head(10)
)
display(top_confusions)

tuning_ax = plot_tuning_candidates(
    tuned.candidates,
    color_by="estimator__depth",
    validation_score=result.metrics.bacc_combined,
)
tuning_ax.set_title("Kombinierte Labels: CV-Score von CatBoost")
display(tuning_ax.figure)

for figure in (
    plot_confusion_matrices(result.y_val, result.y_pred),
    plot_class_recall(result.y_val, result.y_pred),
    plot_top_confusions(result.y_val, result.y_pred),
):
    display(figure)

In [ ]:
from awp2.config import MODEL_DOCS_FIGURES_DIR
from awp2.plots import save_doc_figure

save_doc_figure(tuning_ax.figure, "combined_catboost_tuning", MODEL_DOCS_FIGURES_DIR)

confusion_figure = plot_confusion_matrices(result.y_val, result.y_pred)
save_doc_figure(confusion_figure, "combined_catboost_confusion", MODEL_DOCS_FIGURES_DIR)

recall_figure = plot_class_recall(result.y_val, result.y_pred)
save_doc_figure(recall_figure, "combined_catboost_recall", MODEL_DOCS_FIGURES_DIR)

confusions_figure = plot_top_confusions(result.y_val, result.y_pred)
save_doc_figure(confusions_figure, "combined_catboost_confusions", MODEL_DOCS_FIGURES_DIR)

{
    "tuning_run": tuned.run_id,
    "evaluation_run": result.run_id,
    "invalid_combinations": result.metrics.invalid_combinations,
}